# Phase 4: Generation and Evaluation
## SVG Scaling Laws — CS-GY 6923

**Steps:**
1. Mount Drive & clone/pull repo
2. Install dependencies (incl. cairosvg system libs)
3. Symlink `outputs/` to Drive
4. Verify the SP XL checkpoint and the tokenizer are present
5. Generate samples (script 10)
6. Evaluate samples + test perplexity (script 11)
7. Plot sample figures (script 12)
8. Display the metrics table inline

---
## Cell 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/svg-scaling-laws'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project dir: {DRIVE_DIR}')

---
## Cell 1: Clone / Pull Repository

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/svg-scaling-laws.git'  # <-- EDIT THIS
REPO_DIR = '/content/svg-scaling-laws'

import os
if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest ...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo ...')
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## Cell 2: Install Dependencies
`cairosvg` needs Cairo system libraries on Colab.

In [ ]:
# System packages required by cairosvg (Colab usually has these but be safe)
!apt-get -qq install -y libcairo2 libpango-1.0-0 libpangocairo-1.0-0 > /dev/null 2>&1 || true

!pip install -q -r requirements.txt
!pip install -q cairosvg
print('Dependencies installed.')

---
## Cell 3: Symlink `outputs/` → Drive

In [ ]:
import os, sys, shutil

REPO_DIR      = '/content/svg-scaling-laws'
DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'
LOCAL_OUTPUTS = os.path.join(REPO_DIR, 'outputs')

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

if os.path.islink(LOCAL_OUTPUTS):
    os.unlink(LOCAL_OUTPUTS)
elif os.path.exists(LOCAL_OUTPUTS):
    shutil.rmtree(LOCAL_OUTPUTS)

os.symlink(DRIVE_OUTPUTS, LOCAL_OUTPUTS)
print(f'Symlink: {LOCAL_OUTPUTS} -> {DRIVE_OUTPUTS}')

for d in ['logs', 'plots', 'samples', 'samples/unconditional', 'samples/prefix',
          'samples/rendered', 'samples/rendered/unconditional', 'samples/rendered/prefix']:
    os.makedirs(os.path.join(DRIVE_OUTPUTS, d), exist_ok=True)

sys.path.insert(0, REPO_DIR)
print('Path configured.')

---
## Cell 4: Verify Checkpoint & Tokenizer

In [ ]:
import os
import torch

CKPT_PATH      = 'outputs/checkpoints/xl/best.pt'
TOKENIZER_PATH = 'outputs/tokenizer/tokenizer.json'
TEST_BIN       = 'outputs/data/binary/test.bin'

for label, p in [('checkpoint', CKPT_PATH),
                 ('tokenizer',  TOKENIZER_PATH),
                 ('test bin',   TEST_BIN)]:
    exists = os.path.exists(p)
    sz = os.path.getsize(p) / 1e6 if exists else 0
    print(f'  {label:<11} {p}  exists={exists}  size={sz:.1f} MB')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(
        f'Best XL checkpoint missing at {CKPT_PATH}. '
        'Phase 2 must be complete first.')

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print('\nCheckpoint summary:')
print(f'  step           = {ckpt.get("step", "?")}')
print(f'  best_val_loss  = {ckpt.get("best_val_loss", float("nan")):.4f}')
print(f'  config         = {ckpt["config"]}')

---
## Cell 5: Generate Samples
Runs `scripts/10_generate_samples.py`. Writes:
- `outputs/samples/unconditional/uncond_t{T}_{i}.svg` (15 samples = 3 temps × 5)
- `outputs/samples/prefix/prefix{i}_t{T}.svg` (15 samples = 5 prefixes × 3 temps)
- `outputs/samples/manifest.json`

In [ ]:
!python scripts/10_generate_samples.py \
    --checkpoint outputs/checkpoints/xl/best.pt \
    --tokenizer_dir outputs/tokenizer \
    --out_dir outputs/samples \
    --n_uncond 15 \
    --max_new_tokens 1024 \
    --top_k 50 \
    --top_p 0.95

---
## Cell 6: Evaluate Samples + Test Perplexity
Runs `scripts/11_evaluate_samples.py`. Writes:
- `outputs/samples/rendered/{unconditional,prefix}/*.png` for every renderable sample
- `outputs/logs/evaluation_metrics.json`

In [ ]:
!python scripts/11_evaluate_samples.py \
    --checkpoint outputs/checkpoints/xl/best.pt \
    --samples_dir outputs/samples \
    --rendered_dir outputs/samples/rendered \
    --test_bin outputs/data/binary/test.bin \
    --metrics_path outputs/logs/evaluation_metrics.json \
    --ppl_max_batches 200

---
## Cell 7: Plot Sample Figures
Runs `scripts/12_plot_samples.py`. Writes:
- `outputs/plots/samples_unconditional_grid.png`
- `outputs/plots/samples_temperature_comparison.png`
- `outputs/plots/samples_prefix_completion.png`

In [ ]:
!python scripts/12_plot_samples.py \
    --samples_dir outputs/samples \
    --rendered_dir outputs/samples/rendered \
    --plots_dir outputs/plots \
    --prefix_temperature 0.8

---
## Cell 8: Display Plots Inline

In [ ]:
from IPython.display import Image, display
import os

for fname in ['samples_unconditional_grid.png',
              'samples_temperature_comparison.png',
              'samples_prefix_completion.png']:
    p = os.path.join('outputs/plots', fname)
    print(f'\n=== {fname} ===')
    if os.path.exists(p):
        display(Image(filename=p))
    else:
        print(f'  MISSING: {p}')

---
## Cell 9: Print Evaluation Metrics Table

In [ ]:
import json

with open('outputs/logs/evaluation_metrics.json') as f:
    metrics = json.load(f)

def fmt_block(name, block):
    counts = block['counts']
    rates  = block['rates']
    print(f'\n{name} (n={counts["total"]}):')
    print(f'  XML valid       : {counts["xml_valid"]:>3}/{counts["total"]}  ({rates["xml_valid_rate"]*100:5.1f}%)')
    print(f'  Has <svg> root  : {counts["has_svg_root"]:>3}/{counts["total"]}  ({rates["has_svg_root_rate"]*100:5.1f}%)')
    print(f'  Tags closed     : {counts["tags_closed"]:>3}/{counts["total"]}  ({rates["tags_closed_rate"]*100:5.1f}%)')
    print(f'  SVG renderable  : {counts["svg_renderable"]:>3}/{counts["total"]}  ({rates["svg_renderable_rate"]*100:5.1f}%)')

print('=' * 60)
print('SAMPLE EVALUATION METRICS')
print('=' * 60)
fmt_block('Unconditional', metrics['unconditional'])
fmt_block('Prefix-conditioned', metrics['prefix'])
fmt_block('Combined', metrics['combined'])

if 'perplexity' in metrics:
    p = metrics['perplexity']
    print('\n' + '=' * 60)
    print('TEST-SET PERPLEXITY (best SP XL model)')
    print('=' * 60)
    print(f'  Mean cross-entropy : {p["mean_cross_entropy"]:.4f}')
    print(f'  Perplexity         : {p["perplexity"]:.2f}')
    print(f'  Tokens scored      : {p["n_tokens_scored"]:,}')
print('=' * 60)